In [1]:
!pip install -q datasets transformers accelerate evaluate jiwer librosa soundfile peft bitsandbytes torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.6 MB/s eta 0:00:00


In [2]:
import os
from google.colab import drive
from huggingface_hub import notebook_login

# 1. Монтируем диск
drive.mount('/content/drive')

# 2. Перенаправляем весь кеш на Диск (вместо системного диска Colab)
os.environ["HF_DATASETS_CACHE"] = "/content/drive/MyDrive/huggingface_cache"
os.makedirs("/content/drive/MyDrive/huggingface_cache", exist_ok=True)

# 3. Логин
notebook_login()

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from datasets import load_dataset, concatenate_datasets, Audio
from transformers import WhisperProcessor
import gc

model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id, language="English", task="transcribe")

print("Загрузка датасетов...")
ds1 = load_dataset("Jzuluaga/atcosim_corpus", split="train")
ds2 = load_dataset("adityarra07/ATC_train", split="train")
ds3 = load_dataset("Jzuluaga/uwb_atcc", split="train")
ds4 = load_dataset("pietrobarbera/atcosim-noise", split="train")

def standardize_columns(dataset):
    if "transcription" in dataset.column_names:
        dataset = dataset.rename_column("transcription", "text")
    elif "sentence" in dataset.column_names:
        dataset = dataset.rename_column("sentence", "text")
    elif "transcript" in dataset.column_names:
        dataset = dataset.rename_column("transcript", "text")
    return dataset

# Унификация названий колонок
ds1 = standardize_columns(ds1)
ds2 = standardize_columns(ds2)
ds3 = standardize_columns(ds3)
ds4 = standardize_columns(ds4)

print("Приведение к 16кГц и фильтрация...")
# Унификация аудио и удаление лишних колонок
ds1 = ds1.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "text"])
ds2 = ds2.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "text"])
ds3 = ds3.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "text"])
ds4 = ds4.cast_column("audio", Audio(sampling_rate=16000)).select_columns(["audio", "text"])

print("Склейка и перемешивание...")
combined_dataset = concatenate_datasets([ds1, ds2, ds3, ds4]).shuffle(seed=42)

print("Разделение на Обучение и Тест...")
# Отделяем 400 записей для проверки качества (WER)
dataset_splits = combined_dataset.train_test_split(test_size=400, seed=42)
train_data = dataset_splits["train"]
eval_data = dataset_splits["test"]

# Тотальная очистка оперативной памяти
del ds1, ds2, ds3, ds4, combined_dataset
gc.collect()

print(f"✅ Готово! На обучение: {len(train_data)}, На тест: {len(eval_data)}")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/6.03k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/667 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.31k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/360 [00:00<?, ?B/s]

Приведение к 16кГц и фильтрация...
Склейка и перемешивание...
Разделение на Обучение и Тест...
✅ Готово! На обучение: 48327, На тест: 400


In [4]:
import gc

# 1. Принудительно очищаем мусор перед началом тяжелой работы
gc.collect()

def prepare_dataset(batch):
    audio = batch["audio"]
    # Извлекаем признаки
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=16000).input_features[0]
    # Извлекаем текст
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

print("Векторизация тестовой выборки (400 записей)...")
# Сначала обрабатываем маленькую выборку
mapped_eval_dataset = eval_data.map(
    prepare_dataset,
    remove_columns=["audio", "text"],
    writer_batch_size=100  # <--- Защита ОЗУ №1
)

# Снова чистим память перед гигантским куском
gc.collect()

print("Векторизация обучающей выборки (~44 600 записей)...")
print("Это займет время, но ОЗУ теперь в безопасности.")
# Обрабатываем основную базу
mapped_train_dataset = train_data.map(
    prepare_dataset,
    remove_columns=["audio", "text"],
    writer_batch_size=100  # <--- Защита ОЗУ №2
)

# Финальная очистка
gc.collect()

print("✅ Векторизация успешно завершена без перегрузки памяти.")

Векторизация тестовой выборки (400 записей)...


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Векторизация обучающей выборки (~44 600 записей)...
Это займет время, но ОЗУ теперь в безопасности.


Map:   0%|          | 0/48327 [00:00<?, ? examples/s]

✅ Векторизация успешно завершена без перегрузки памяти.


In [5]:
import torch
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

# Класс выравнивания длинных и коротких аудио
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in features], return_tensors="pt")
        labels["input_ids"] = labels["input_ids"].masked_fill(labels.attention_mask.ne(1), -100)
        batch["labels"] = labels["input_ids"][:, 1:] if (labels["input_ids"][:, 0] == self.processor.tokenizer.bos_token_id).all() else labels["input_ids"]
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Загружаем метрику Word Error Rate
metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Заменяем -100 обратно на pad_token_id для декодирования
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # Превращаем цифры обратно в текст
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Считаем процент ошибок
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [6]:
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

model = WhisperForConditionalGeneration.from_pretrained(model_id, device_map="auto")
model.config.use_cache = False # Обязательно выключить при обучении

config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, config)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

trainable params: 3,538,944 || all params: 245,273,856 || trainable%: 1.4429


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import os

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/Aviation_Whisper_V3",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-4,
    warmup_steps=100,
    max_steps=5000,
    fp16=True,
    eval_strategy="steps",                # ИСПРАВЛЕНО: evaluation_strategy -> eval_strategy
    eval_steps=100,                       # Проверка качества
    save_steps=100,                       # Сохранение чекпоинта
    save_total_limit=3,                   # Хранение 3 последних чекпоинтов
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=225,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=mapped_train_dataset,
    eval_dataset=mapped_eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Логика автоматического подхвата чекпоинтов при перезапусках
latest_checkpoint = None
if os.path.isdir(training_args.output_dir):
    checkpoints = [os.path.join(training_args.output_dir, d) for d in os.listdir(training_args.output_dir) if d.startswith("checkpoint")]
    if checkpoints:
        latest_checkpoint = max(checkpoints, key=os.path.getmtime)

if latest_checkpoint:
    print(f"Возобновление обучения с чекпоинта: {latest_checkpoint}")
    trainer.train(resume_from_checkpoint=latest_checkpoint)
else:
    print("Старт обучения с нуля...")
    trainer.train()

# Финальное сохранение
model.save_pretrained("/content/drive/MyDrive/Aviation_Whisper_V3_FINAL")
processor.save_pretrained("/content/drive/MyDrive/Aviation_Whisper_V3_FINAL")

Старт обучения с нуля...


Step,Training Loss,Validation Loss,Wer
100,2.779972,0.807454,30.699625
200,1.234597,0.582572,38.225557
300,1.122919,0.500064,18.229971
400,12.851013,7.610262,100.000000
500,7.529663,1.926951,46.877069
600,0.930396,0.437317,45.221805
700,0.882863,0.414456,34.561907
800,0.768982,0.380492,29.971309
900,0.749033,0.371848,41.822997
1000,0.747972,0.342435,43.080998


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its para